# 00 — Setup & downloads

Run once per pod. Run order: **00 → 01 → 02 → 03 → 04 → 05 → 06**.

Before you start:
- Run inside a PyTorch template (or your `preprocess_5090` venv). The install cell pins torch 2.8.0/cu128.
- `export HF_TOKEN=hf_...` (RunPod env var) — the NVIDIA PersonaPlex repo (mimi, SentencePiece, voice prompts) is gated; you must have accepted its license. The EmotionPlex dataset itself is public.
- Upload your local `IMTalker/prompts/RB_Robert_System_Prompt_full.txt` to `<root>/prompts/` (checked below).
- **License note**: the EmotionPlex dataset audio is Higgs-generated — research / non-commercial use only.
- Disk: ~45 GB (bnb4 ~6 GB, dataset ~28 GB, emotionplex-7b ~17 GB one-off).

**Run mode**: the first code cell sets `SMOKE_TEST` (default `False`).
`True` = fast end-to-end verification of the whole pipeline (~52 stratified
clips, 2-epoch trainings, 12 generation samples; ~15–25 min with cached
checkpoints). All derived artifacts go to `<root>/_smoke/` and can never touch
full-run artifacts (checkpoints, prompts and the raw dataset stay shared).
To switch modes: edit the flag, re-run that ONE cell (it rewrites
`run_config.json`), then **restart the kernels of the other notebooks** — they
read the mode via `eh.get_paths()` when their first cell runs.


In [ ]:
# ============================ RUN MODE ============================
# SMOKE_TEST = False -> the real full pipeline (default).
# SMOKE_TEST = True  -> tiny end-to-end verification run; outputs isolated in
#                       <root>/_smoke/ (full-run artifacts untouched).
# This cell is stdlib-only and takes ~1s: re-run it alone to switch modes.
SMOKE_TEST = False

import json, os
from pathlib import Path
_root = Path(os.environ.get("EMOTION_HEAD_ROOT", "/workspace/emotion_head"))
_root.mkdir(parents=True, exist_ok=True)
(_root / "run_config.json").write_text(
    json.dumps({"smoke_test": bool(SMOKE_TEST), "version": 1}, indent=2))
print(f"run mode = {'SMOKE_TEST' if SMOKE_TEST else 'FULL'} "
      f"-> {_root / 'run_config.json'}")
print("other notebooks (and GPU workers) pick the mode up via eh.get_paths(); "
      "restart their kernels after switching")


In [ ]:
import sys, torch
print(sys.version)
print("torch:", torch.__version__, "cuda:", torch.version.cuda)
assert torch.cuda.is_available(), "GPU pod required"
print("gpu:", torch.cuda.get_device_name(0),
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")


In [ ]:
DO_INSTALL = True   # False if your env already matches (e.g. preprocess_5090)
if DO_INSTALL:
    %pip install -q torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
    %pip install -q bitsandbytes sphn sentencepiece soundfile safetensors einops num2words "huggingface_hub[hf_transfer]" scikit-learn matplotlib pandas pyarrow tqdm ipywidgets
import torch
print("torch:", torch.__version__)


In [ ]:
from pathlib import Path
import os, sys, json, time
import numpy as np
import torch

sys.path.insert(0, os.getcwd())
import eh_common as eh

P = eh.get_paths()   # override: P = eh.get_paths("/your/root") or set EMOTION_HEAD_ROOT
eh.mode_banner(P)    # FULL vs SMOKE_TEST — set once in NB00 (run_config.json)
RUN = eh.run_profile(P)   # all size knobs (full | smoke) — see eh_common.run_profile
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16
print("ROOT:", P.ROOT, "| device:", DEVICE)
for k in ("MOSHI_BNB4", "MIMI_FILE", "SPM_FILE", "VOICE_PT"):
    p = Path(getattr(P, k))
    print(f"  {k}: {'OK' if p.exists() else 'MISSING'}  {p}")


In [ ]:
import os
from huggingface_hub import login, whoami
tok = os.environ.get("HF_TOKEN", "").strip()
assert tok, "HF_TOKEN env var is required (gated repos). Set it in the pod env or: %env HF_TOKEN=hf_..."
login(token=tok, add_to_git_credential=False)
print("logged in as:", whoami()["name"])
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")


In [ ]:
import shutil, tarfile
from huggingface_hub import snapshot_download, hf_hub_download

P.CKPT.mkdir(parents=True, exist_ok=True)
snapshot_download(eh.BNB4_REPO, local_dir=str(P.CKPT),
                  allow_patterns=["model_bnb_4bit.pt", "moshi/**"])
print("bnb4 snapshot done (quantized LM + moshi source)")

# The bnb4 community repo ships ONLY the quantized LM. Mimi, SentencePiece and
# the voice prompts live in the gated NVIDIA repo (needs HF_TOKEN + license).
eh.hf_fetch(eh.NVIDIA_REPO, "tokenizer-e351c8d8-checkpoint125.safetensors", P.MIMI_FILE)
eh.hf_fetch(eh.NVIDIA_REPO, "tokenizer_spm_32k_3.model", P.SPM_FILE)

if not P.VOICE_PT.exists():
    print("VARM3.pt not in bnb4 snapshot; fetching voices.tgz from", eh.NVIDIA_REPO)
    tgz = Path(hf_hub_download(eh.NVIDIA_REPO, "voices.tgz"))
    voices_dir = tgz.parent / "voices"
    if not voices_dir.exists():
        with tarfile.open(tgz, "r:gz") as tar:
            tar.extractall(path=tgz.parent)
    P.VOICE_PT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(voices_dir / "VARM3.pt", P.VOICE_PT)

ok = True
for k in ("MOSHI_BNB4", "MIMI_FILE", "SPM_FILE", "VOICE_PT"):
    p = Path(getattr(P, k))
    good = p.exists()
    ok &= good
    print(("OK      " if good else "MISSING "), p,
          f"{p.stat().st_size/1e9:.2f} GB" if good else "")
assert ok, "checkpoint assets missing"


In [ ]:
import subprocess, sys
if not (P.MOSHI_ROOT / "pyproject.toml").exists():
    print("moshi source missing -> fetching moshi/** from", eh.BNB4_REPO)
    snapshot_download(eh.BNB4_REPO, local_dir=str(P.CKPT), allow_patterns=["moshi/**"])
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                    str(P.MOSHI_ROOT), "--no-deps"])
print("pip editable install rc:", r.returncode,
      "(non-zero is fine: sys.path fallback below still makes moshi importable)")
print("moshi importable:", eh.ensure_moshi_importable(P))


In [ ]:
P.PROMPTS.mkdir(parents=True, exist_ok=True)
if not P.TEXT_PROMPT_FILE.exists():
    print("!! ACTION REQUIRED: upload your local file")
    print("   IMTalker/prompts/RB_Robert_System_Prompt_full.txt")
    print("   to", P.TEXT_PROMPT_FILE)
else:
    txt = P.TEXT_PROMPT_FILE.read_text(encoding="utf-8", errors="replace")
    print(f"text prompt OK: {len(txt)} chars | {txt[:100]!r} ...")


In [ ]:
import re
from huggingface_hub import HfApi, hf_hub_download
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

api = HfApi()
files = api.list_repo_files(eh.DATASET_REPO, repo_type="dataset")
P.DATA.mkdir(parents=True, exist_ok=True)
(P.DATA / "_repo_files.json").write_text(json.dumps(files))
json_files = sorted(f for f in files if f.endswith((".json", ".jsonl")))
audio_files = sorted(f for f in files if f.endswith((".wav", ".tar", ".tar.gz")))
print(f"{len(files)} files | {len(json_files)} json/jsonl | {len(audio_files)} audio")
print("sample audio names:", [f.split("/")[-1] for f in audio_files[:5]])

def _dl(f):
    return hf_hub_download(eh.DATASET_REPO, f, repo_type="dataset", local_dir=str(P.DATA))

def _dl_all(fs, desc, workers=16):
    with ThreadPoolExecutor(max_workers=workers) as ex:
        list(tqdm(ex.map(_dl, fs), total=len(fs), desc=desc))

if P.SMOKE:
    # SMOKE: top-level manifests + sidecars of a small deterministic candidate
    # pool only (~130 files instead of ~24k), then a stratified final pick.
    _dl_all([f for f in json_files if "/" not in f], "manifests")

    def _pool_rows(manifest, n):
        rows = [json.loads(l) for l in open(P.DATA / manifest, encoding="utf-8")
                if l.strip()]
        rng = np.random.RandomState(eh.SEED)
        idx = sorted(rng.permutation(len(rows))[: min(n, len(rows))])
        return [rows[i] for i in idx]

    pool_tr = _pool_rows("train.jsonl", RUN.smoke_sidecar_candidates_train)
    pool_va = _pool_rows("val.jsonl", RUN.smoke_sidecar_candidates_val)
    _dl_all(sorted({str(r["path"]).replace(".wav", ".json") for r in pool_tr + pool_va}),
            "sidecars (smoke pool)")

    def _emo_of(wav_rel):
        # agent emotion: sidecar text_prompt "Answer with a {e} tone."
        try:
            d = json.loads((P.DATA / wav_rel.replace(".wav", ".json"))
                           .read_text(encoding="utf-8"))
            m = re.search(r"a (happy|sad|angry|neutral) tone",
                          str(d.get("text_prompt", "")).lower())
            if m:
                return m.group(1)
        except Exception:
            pass
        parts = Path(wav_rel).stem.split("_")       # "<user>_<agent>_NNN"
        return parts[1] if len(parts) > 1 and parts[1] in eh.EMO2ID else None

    rng = np.random.RandomState(eh.SEED)
    chosen = []
    for rows, per_emo in ((pool_tr, RUN.smoke_train_clips_per_emotion),
                          (pool_va, RUN.smoke_val_clips_per_emotion)):
        by_emo = {e: [] for e in eh.EMOTIONS}
        for r in rows:
            e = _emo_of(str(r.get("path", "")))
            if e:
                by_emo[e].append(str(r["path"]))
        for e in eh.EMOTIONS:
            cand = sorted(by_emo[e])
            take = [cand[i] for i in rng.permutation(len(cand))[: min(per_emo, len(cand))]]
            chosen += take
            print(f"  smoke pool {e}: picked {len(take)}/{len(cand)}")
    SMOKE_WAVS = sorted(set(chosen))
    assert SMOKE_WAVS, "smoke subset selection failed — no clips chosen"
    P.OUT.mkdir(parents=True, exist_ok=True)
    (P.OUT / "smoke_subset.json").write_text(json.dumps(
        dict(clips=sorted({Path(w).stem for w in SMOKE_WAVS}), wavs=SMOKE_WAVS),
        indent=2))
    print(f"smoke subset: {len(SMOKE_WAVS)} wavs -> {P.OUT / 'smoke_subset.json'}")
else:
    _dl_all(json_files, "sidecars")
print("sidecars + manifests done")


In [ ]:
DOWNLOAD_ALL_AUDIO = RUN.nb00_download_all_audio   # full True | SMOKE False
MAX_CLIPS_SUBSET = RUN.nb00_max_clips_subset       # (only if DOWNLOAD_ALL_AUDIO=False by hand)

if P.SMOKE:
    targets = SMOKE_WAVS     # stratified subset chosen in the sidecar cell
    print(f"SMOKE: downloading only the {len(targets)} subset wavs "
          f"(~{len(targets)} files, a few tens of MB)")
elif DOWNLOAD_ALL_AUDIO:
    targets = audio_files
else:
    rows = []
    for jf in sorted(P.DATA.rglob("*.jsonl")):
        for line in open(jf, encoding="utf-8"):
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except Exception:
                    pass
    rng = np.random.RandomState(eh.SEED)
    idx = rng.permutation(len(rows))[:MAX_CLIPS_SUBSET]
    wanted = set()
    for i in idx:
        for key in ("path", "wav", "audio", "file"):
            v = rows[i].get(key)
            if v:
                wanted.add(str(v).split("/")[-1])
    targets = [f for f in audio_files if any(w in f for w in wanted)]
    if not targets:
        print("!! subset matching found nothing; falling back to ALL audio files")
        targets = audio_files
print(f"downloading {len(targets)} audio files")
with ThreadPoolExecutor(max_workers=8) as ex:
    list(tqdm(ex.map(_dl, targets), total=len(targets), desc="audio"))
print("wav on disk:", sum(1 for _ in P.DATA.rglob("*.wav")))


In [ ]:
import gc
sp = eh.load_sp(P)
mimi = eh.load_mimi(P, DEVICE)
lm = eh.load_lm_bnb4(P, DEVICE, DTYPE)
print(f"LM params: {sum(p.numel() for p in lm.parameters())/1e9:.2f}B | "
      f"cuda mem: {torch.cuda.memory_allocated()/1e9:.2f} GiB")
lm_gen = eh.make_lm_gen(lm, DEVICE)
eh.install_hidden_step(lm_gen)
sil = np.zeros(20 * eh.FRAME_SAMPLES, np.float32)
codes = eh.mimi_encode_codes(mimi, sil, DEVICE)
assert codes.shape == (20, 8), codes.shape
for t in range(20):
    # _step_force tolerates the offset-0 init step (first call returns None and
    # discards its inputs in moshi's prepare_step_input; re-feeding is exact).
    out = eh._step_force(lm_gen, input_tokens=codes[t].view(1, 8, 1).to(DEVICE),
                         text_token=eh.TEXT_PAD)
    assert out[2].shape[-1] == eh.HELIUM_DIM, out[2].shape
lat = eh.codes_to_latent(mimi, codes, DEVICE)
assert lat.shape == (20, 512), lat.shape
print("SMOKE TEST OK — bnb4 loads, live step path works, hidden [4096] + latent [512] confirmed")
del lm, lm_gen, mimi
gc.collect()
torch.cuda.empty_cache()


In [ ]:
marker = dict(time=time.time(), torch=torch.__version__, cuda=torch.version.cuda,
              gpu=torch.cuda.get_device_name(0), smoke_test=bool(P.SMOKE))
P.ROOT.mkdir(parents=True, exist_ok=True)
(P.ROOT / "_env_ok.json").write_text(json.dumps(marker, indent=2))
if P.SMOKE:
    print("SMOKE mode ready: run 01 → 06 with fresh kernels to verify the whole "
          "pipeline end-to-end (artifacts land in", P.OUT, ")")
print("Setup complete → continue with 01_dataset_inspect_and_manifest.ipynb")
